In [0]:
import numpy as np
import pandas as pd


def clean_evictions_data(
    input_path: str = "cook_county_evictions_combined_2000_2018.csv",
    output_path: str = "cook_county_evictions_cleaned_2001_2015.csv",
) -> pd.DataFrame:
    # 1. Load with zero-padded string FIPS / GEOID identifiers
    df = pd.read_csv(
        input_path,
        dtype={"cofips": str, "geoid": str},
        low_memory=False,
    )
    df["cofips"] = df["cofips"].str.zfill(5)
    df["geoid"] = df["geoid"].str.zfill(11)

    # 2. Remove exact duplicates and enforce unique (geoid, year) panel keys
    df = df.drop_duplicates().drop_duplicates(subset=["geoid", "year"]).copy()

    # 3. Drop non-residential / special-use tracts (zero-pop, airports, Lake Michigan)
    non_residential_geoids = {
        "17031381700",  # Census Tract 3817 (0 population)
        "17031980000",  # Census Tract 9800 (O'Hare International Airport)
        "17031980100",  # Census Tract 9801 (Midway International Airport)
        "17031990000",  # Census Tract 9900 (Lake Michigan water tract)
    }
    df = df[~df["geoid"].isin(non_residential_geoids)].copy()

    # 4. Preserve Chicago vs. suburban Cook County indicator prior to imputing cha_*
    df["is_chicago_tract"] = df["cha_tract_name"].notna()

    # 5. Backfill missing cha_tract_name ("Census Tract X" -> "Tract X")
    df["cha_tract_name"] = df["cha_tract_name"].fillna(
        df["tract"].str.replace("Census ", "", regex=False)
    )

    # 6. Set judgements & judgement_rate to 0 where filings == 0 before median imputation
    #    (avoids creating logical violations where imputed judgements > 0 filings)
    zero_filings = df["filings"] == 0
    df.loc[zero_filings & df["judgements"].isna(), "judgements"] = 0.0
    df.loc[zero_filings & df["judgement_rate"].isna(), "judgement_rate"] = 0.0

    # 7. Convert +inf / -inf to NaN, then impute all remaining numeric nulls with medians
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)
    df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

    # 8. Cast discrete count/year columns to int64 and sort panel order
    int_cols = ["year", "filings", "threatened", "judgements"]
    df[int_cols] = df[int_cols].round().astype("int64")
    df = df.sort_values(["geoid", "year"]).reset_index(drop=True)

    # 9. Validate before saving
    assert df.isna().sum().sum() == 0, "Remaining NaN values found!"
    assert np.isinf(df[numeric_cols]).sum().sum() == 0, "Remaining Inf values found!"
    assert (df["judgements"] <= df["filings"]).all(), "judgements > filings violation!"
    assert (df["threatened"] <= df["filings"]).all(), "threatened > filings violation!"

    df.to_csv(output_path, index=False)
    return df


if __name__ == "__main__":
    cleaned_df = clean_evictions_data()
    print(f"Saved cleaned dataset: {cleaned_df.shape[0]:,} rows x {cleaned_df.shape[1]} columns")